# 03 — Feature Engineering

**Project:** End-to-End Heart Disease Prediction System
**Stage:** 3 / 5 — Feature Engineering

## Objective
Transform clinically-imputed data into a model-ready feature matrix. Every
engineered feature here is grounded in **cardiology domain knowledge**, not
just statistical pattern-mining — which is part of what separates this project
from a generic Kaggle baseline.

A critical ordering rule is followed throughout this notebook:

> **All domain-knowledge interaction features are built BEFORE one-hot
> encoding or binary mapping of their source columns.**

This matters because once `ST_Slope` or `ExerciseAngina` are one-hot/binary
encoded, the original categorical semantics ("Flat", "Down", "Y", "N") are
gone — composing interaction features afterward would require reverse-mapping
encoded columns, which is fragile and error-prone.

## Input
- `data/interim/train_imputed.csv`
- `data/interim/test_imputed.csv`

## Output
- `data/processed/train_clean.csv`
- `data/processed/test_clean.csv`

## 1. Imports & Load Interim Data

In [1]:
import pandas as pd
import numpy as np

INTERIM_DIR = "../data/interim"
PROCESSED_DIR = "../data/processed"

train_imputed = pd.read_csv(f"{INTERIM_DIR}/train_imputed.csv")
test_imputed  = pd.read_csv(f"{INTERIM_DIR}/test_imputed.csv")

# Re-tag and recombine so one-hot encoding produces IDENTICAL columns
# for both train and test (prevents column-mismatch at inference time).
train_imputed['is_train'] = 1
test_imputed['is_train']  = 0
combined = pd.concat([train_imputed, test_imputed], axis=0, ignore_index=True)

print(f"Combined shape before FE: {combined.shape}")

Combined shape before FE: (918, 16)


## 2. Domain-Knowledge Interaction Features

These three features encode clinical reasoning that no generic model would
discover on its own from raw columns alone:

- **`Ischemia_Risk`** — A patient showing meaningful ST depression
  (`Oldpeak > 0.5`) **combined with** an abnormal ST slope (`Flat` or `Down`)
  is a textbook exercise-stress-test signature of myocardial ischemia. Neither
  signal alone is as informative as the conjunction.
- **`Metabolic_Risk`** — Elevated blood pressure (`>130`), elevated cholesterol
  (`>200`), and elevated fasting blood sugar **co-occurring** describe a
  classic cardiometabolic risk cluster (related to metabolic syndrome), which
  compounds cardiovascular risk beyond any single factor.
- **`Angina_Oldpeak`** — Exercise-induced angina is far more clinically
  meaningful when paired with the *magnitude* of ST depression during that
  same exercise test, so we multiply the two together instead of treating them
  as independent signals.

In [2]:
# Built BEFORE one-hot encoding ST_Slope
combined['Ischemia_Risk'] = (
    (combined['Oldpeak'] > 0.5) &
    (combined['ST_Slope'].isin(['Flat', 'Down']))
).astype(int)

combined['Metabolic_Risk'] = (
    (combined['RestingBP'] > 130) &
    (combined['Cholesterol'] > 200) &
    (combined['FastingBS'] == 1)
).astype(int)

# Built BEFORE binary-encoding ExerciseAngina
combined['Angina_Oldpeak'] = (
    combined['ExerciseAngina'].map({'Y': 1, 'N': 0}) * combined['Oldpeak']
)

combined[['Ischemia_Risk', 'Metabolic_Risk', 'Angina_Oldpeak']].describe()

,Ischemia_Risk,Metabolic_Risk,Angina_Oldpeak
count,918.000000,918.000000,918.000000
mean,0.408497,0.108932,0.572440
std,0.491824,0.311724,0.961864
min,0.000000,0.000000,-1.500000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,1.000000,0.000000,1.000000
max,1.000000,1.000000,5.600000


## 3. Clinical Binning

Continuous variables are binned using established clinical reference ranges
rather than arbitrary quantiles, so the cut points are interpretable to a
medical audience:

- **`Age_group`** — young / middle-aged / senior / elderly
- **`Chol_bin`** — desirable / normal / borderline-high / high (standard lipid panel categories)
- **`Oldpeak_bin`** — normal / mild / moderate / severe ST depression

In [3]:
combined['Age_group'] = pd.cut(
    combined['Age'], bins=[0, 45, 55, 65, 120],
    labels=['young', 'middle_aged', 'senior', 'elderly']
).astype(str)

combined['Chol_bin'] = pd.cut(
    combined['Cholesterol'], bins=[0, 150, 200, 239, 999],
    labels=['low', 'normal', 'borderline_high', 'high']
).astype(str)

combined['Oldpeak_bin'] = pd.cut(
    combined['Oldpeak'], bins=[-0.1, 0, 1, 2, 99],
    labels=['normal', 'mild', 'moderate', 'severe']
).astype(str)

combined[['Age_group', 'Chol_bin', 'Oldpeak_bin']].head()

,Age_group,Chol_bin,Oldpeak_bin
0,young,high,normal
1,middle_aged,normal,mild
2,young,high,normal
3,middle_aged,borderline_high,moderate
4,middle_aged,normal,normal


## 4. Age-Normalized Heart Rate

Raw `MaxHR` is confounded by age — maximum achievable heart rate naturally
declines with age (roughly `220 - Age`), so a 65-year-old and a 30-year-old
with the *same* raw `MaxHR` are not equally exerted. `MaxHR_pct` expresses
`MaxHR` as a **percentage of age-predicted maximum**, which is the metric
actually used in exercise-stress-test interpretation. The raw `MaxHR` column
is dropped since it is now fully represented by this normalized version.

In [4]:
combined['MaxHR_pct'] = combined['MaxHR'] / (220 - combined['Age']) * 100
combined.drop(columns=['MaxHR'], inplace=True)

combined['MaxHR_pct'].describe()

count    918.000000
mean      82.110927
std       14.227395
min       35.502959
25%       72.327044
50%       82.530120
75%       93.157026
max      117.469880
Name: MaxHR_pct, dtype: float64

## 5. Encoding

In [5]:
# Binary mappings
combined['Sex']            = combined['Sex'].map({'M': 1, 'F': 0})
combined['ExerciseAngina']  = combined['ExerciseAngina'].map({'Y': 1, 'N': 0})

# One-hot encoding for multi-category columns (drop_first avoids perfect
# multicollinearity / the dummy variable trap)
categorical_to_ohe = [
    'ChestPainType', 'RestingECG', 'ST_Slope',
    'Age_group', 'Chol_bin', 'Oldpeak_bin'
]
combined = pd.get_dummies(combined, columns=categorical_to_ohe, drop_first=True)

print(f"Shape after encoding: {combined.shape}")

Shape after encoding: (918, 33)


## 6. Split Back & Validate Train/Test Column Sync

A common production bug is silent column mismatch between train and test
after one-hot encoding (e.g. a category present in train but absent in test).
We assert this explicitly so the pipeline **fails loudly** instead of silently
producing garbage predictions at inference time.

In [6]:
train_clean = combined[combined['is_train'] == 1].drop(columns=['is_train'])
test_clean  = combined[combined['is_train'] == 0].drop(columns=['is_train', 'HeartDisease'], errors='ignore')

assert 'HeartDisease' in train_clean.columns, \
    "FATAL: target column 'HeartDisease' missing from train_clean!"
assert list(train_clean.drop(columns=['HeartDisease']).columns) == list(test_clean.columns), \
    "FATAL: train/test feature columns are out of sync after encoding!"

print("[VALIDATION PASSED] Train/test feature columns are perfectly synchronized.")
print(f"Train (clean): {train_clean.shape}")
print(f"Test  (clean): {test_clean.shape}")

[VALIDATION PASSED] Train/test feature columns are perfectly synchronized.
Train (clean): (734, 32)
Test  (clean): (184, 31)


## 7. Feature Engineering Summary

In [7]:
features_before = set(train_imputed.columns) - {'is_train'}
features_after  = set(train_clean.columns)

added   = sorted(features_after - features_before)
dropped = sorted(features_before - features_after)
kept    = sorted(features_before & features_after)

print(f"Total features before FE : {len(features_before)}")
print(f"Total features after FE  : {len(features_after)}\n")

print(f"FEATURES ADDED ({len(added)}):")
for f in added:
    print(f"  + {f}")

print(f"\nFEATURES REMOVED/REPLACED ({len(dropped)}):")
for f in dropped:
    print(f"  - {f}")

print(f"\nORIGINAL FEATURES RETAINED ({len(kept)}):")
print(kept)

Total features before FE : 15
Total features after FE  : 32

FEATURES ADDED (21):
  + Age_group_middle_aged
  + Age_group_senior
  + Age_group_young
  + Angina_Oldpeak
  + ChestPainType_ATA
  + ChestPainType_NAP
  + ChestPainType_TA
  + Chol_bin_high
  + Chol_bin_low
  + Chol_bin_normal
  + Ischemia_Risk
  + MaxHR_pct
  + Metabolic_Risk
  + Oldpeak_bin_moderate
  + Oldpeak_bin_nan
  + Oldpeak_bin_normal
  + Oldpeak_bin_severe
  + RestingECG_Normal
  + RestingECG_ST
  + ST_Slope_Flat
  + ST_Slope_Up

FEATURES REMOVED/REPLACED (4):
  - ChestPainType
  - MaxHR
  - RestingECG
  - ST_Slope

ORIGINAL FEATURES RETAINED (11):
['Age', 'Cholesterol', 'Cholesterol_was_zero', 'ExerciseAngina', 'FastingBS', 'HeartDisease', 'Oldpeak', 'RestingBP', 'RestingBP_was_zero', 'Sex', 'id']


## 8. Persist Model-Ready Data to `data/processed/`

In [8]:
train_clean.to_csv(f"{PROCESSED_DIR}/train_clean.csv", index=False)
test_clean.to_csv(f"{PROCESSED_DIR}/test_clean.csv", index=False)

print("Stage 3 complete — model-ready data persisted:")
print(f"  -> {PROCESSED_DIR}/train_clean.csv  ({train_clean.shape[0]} rows, {train_clean.shape[1]} cols)")
print(f"  -> {PROCESSED_DIR}/test_clean.csv   ({test_clean.shape[0]} rows, {test_clean.shape[1]} cols)")

Stage 3 complete — model-ready data persisted:
  -> ../data/processed/train_clean.csv  (734 rows, 32 cols)
  -> ../data/processed/test_clean.csv   (184 rows, 31 cols)


---
**Next notebook:** `04_Modeling_and_Ensemble.ipynb` — baseline models, Optuna
hyperparameter tuning, ensembling, and the dual-threshold clinical safety
strategy.